# Liu2024 EEGNet Baseline

**Goal:** Compact supervised deep learning baseline for Liu2024 acute stroke MI.
Fills the missing deep-model comparison recommended in the pilot reports.

**Why EEGNet?**
- Designed specifically for EEG with very few parameters (~2K).
- Works with 40 trials per subject.
- Standard benchmark in BCI literature (Lawhern et al., 2018).

**Leakage controls:**
- Preprocessing uses fixed transforms (resample, bandpass, average reference) — safe before splitting.
- Per-trial z-score normalisation is trial-wise (no fitting) — safe.
- No scaler, PCA, or covariance mean is fitted on pooled data.
- Early stopping uses an internal validation split from the training fold only.
- Augmentation (if enabled) is applied only to training batches at runtime.

## 1. Imports

In [ ]:
import os
import re
import sys
import json
import copy
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from scipy.io import loadmat

import mne
mne.set_log_level("WARNING")

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

warnings.filterwarnings('ignore', category=RuntimeWarning)
print(f"torch {torch.__version__}, numpy {np.__version__}")

def resolve_device(cfg_device="auto"):
    if cfg_device != "auto":
        return torch.device(cfg_device)
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

## 2. CONFIG

In [ ]:
CONFIG = {
    # --- Paths ---
    "data_root":    "../../liu2024_data/liu2024_figshare/sourcedata",
    "artifact_root": "../../artifacts/liu2024_eegnet_baseline",

    # --- Dataset ---
    "subjects":     "all",
    "random_state": 2026,
    "sfreq_raw":    500,
    "sfreq_model":  128,
    "mi_window_s":  (0.0, 4.0),   # seconds after MI onset
    "bandpass_hz":  (0.5, 40.0),

    # --- Cross-validation ---
    "n_repeats":    10,
    "test_size":    0.40,

    # --- Training ---
    "batch_size":   8,
    "max_epochs":   200,
    "patience":     30,
    "lr":           1e-3,
    "weight_decay": 1e-4,
    "device":       "auto",

    # --- EEGNet hyperparameters ---
    # F1: number of temporal filters
    # D:  depth multiplier (spatial filters per temporal filter)
    # F2: number of pointwise filters = F1 * D
    # dropout_rate: applied after depthwise conv and separable conv
    "F1":           8,
    "D":            2,
    "dropout_rate": 0.5,

    # --- Augmentation (disabled by default) ---
    "augmentation": {
        "enabled":              False,
        "noise_std_fraction":   0.03,   # std as fraction of per-trial std
        "max_shift_samples":    6,       # temporal jitter ± this many samples
        "channel_dropout_prob": 0.05,   # prob of zeroing one channel per trial
    },
}

DATA_ROOT    = Path(CONFIG["data_root"])
ARTIFACT_ROOT = Path(CONFIG["artifact_root"])
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE       = resolve_device(CONFIG["device"])
SFREQ_RAW    = CONFIG["sfreq_raw"]
SFREQ_MODEL  = CONFIG["sfreq_model"]
MI_START_S   = CONFIG["mi_window_s"][0]
MI_STOP_S    = CONFIG["mi_window_s"][1]
WINDOW_SAMPLES = int((MI_STOP_S - MI_START_S) * SFREQ_MODEL)  # 4s * 128 = 512

np.random.seed(CONFIG["random_state"])
random.seed(CONFIG["random_state"])
torch.manual_seed(CONFIG["random_state"])

print(f"Device:        {DEVICE}")
print(f"Data root:     {DATA_ROOT}")
print(f"Artifacts:     {ARTIFACT_ROOT}")
print(f"Window:        {MI_START_S}–{MI_STOP_S} s → {WINDOW_SAMPLES} samples at {SFREQ_MODEL} Hz")
print(f"Augmentation:  {CONFIG['augmentation']['enabled']}")

## 3. Liu2024 Channel Constants

In [ ]:
SOURCE_EEG_NAMES_30 = [
    "Fp1", "Fp2", "Fz", "F3", "F4", "F7", "F8", "FCz", "FC3", "FC4",
    "FT7", "FT8", "Cz", "C3", "C4", "T3", "T4", "CPz",
    "CP3", "CP4", "TP7", "TP8", "Pz", "P3", "P4", "T5", "T6", "Oz", "O1", "O2",
]
CPZ_IDX      = 17
EEG_KEEP_IDX = [i for i in range(30) if i != CPZ_IDX]
EEG_NAMES    = [SOURCE_EEG_NAMES_30[i] for i in EEG_KEEP_IDX]
N_CHANS      = len(EEG_KEEP_IDX)  # 29
print(f"N channels: {N_CHANS}")

## 4. Data Loading and Preprocessing

In [ ]:
def subject_id_from_path(path):
    m = re.search(r"sub[-_ ]?(\d{1,2})", str(path), flags=re.IGNORECASE)
    return int(m.group(1)) if m else int(re.findall(r"\d+", Path(path).stem)[-1])


def preprocess_subject(mat_path):
    """
    Load Liu2024 source .mat → X (40, 29, WINDOW_SAMPLES) float32, y (40,) int.
    Preprocessing: average reference, resample to 128 Hz, bandpass 0.5–40 Hz.
    Per-trial z-score is applied later inside each fold (trial-wise, no leakage).
    """
    mat = loadmat(str(mat_path), squeeze_me=True, struct_as_record=False)
    raw_data = mat.get("rawdata", mat.get("data", None))
    if raw_data is None:
        for k, v in mat.items():
            if not k.startswith("__") and isinstance(v, np.ndarray) and v.ndim == 3:
                raw_data = v; break

    raw_data = np.asarray(raw_data, dtype=np.float64)
    trial_ax = next(ax for ax, sz in enumerate(raw_data.shape) if sz == 40)
    raw_data = np.moveaxis(raw_data, trial_ax, 0)
    if raw_data.shape[1] != 33 and raw_data.shape[2] == 33:
        raw_data = raw_data.transpose(0, 2, 1)
    assert raw_data.shape == (40, 33, 4000), f"Unexpected shape: {raw_data.shape}"

    labels = mat.get("labels", mat.get("label", None))
    y = np.asarray(labels, dtype=int).ravel()
    if set(np.unique(y).tolist()).issubset({1, 2}):
        y = y - 1

    eeg = raw_data[:, EEG_KEEP_IDX, :].astype(np.float64)  # 40 x 29 x 4000

    # MNE pipeline
    n_trials, n_ch, n_t = eeg.shape
    continuous = eeg.reshape(n_ch, n_trials * n_t) * 1e-6  # Volts
    info_raw = mne.create_info(ch_names=EEG_NAMES, sfreq=float(SFREQ_RAW), ch_types=["eeg"] * N_CHANS)
    raw_mne  = mne.io.RawArray(continuous, info_raw, verbose=False)
    raw_mne.set_eeg_reference("average", projection=False, verbose=False)
    raw_mne.resample(SFREQ_MODEL, npad="auto", verbose=False)
    bp_low, bp_high = CONFIG["bandpass_hz"]
    raw_mne.filter(bp_low, bp_high, method="fir", phase="zero", verbose=False)

    data_r = raw_mne.get_data() * 1e6  # microvolts
    n_t_r  = int(n_t * SFREQ_MODEL / SFREQ_RAW)
    data_trials = data_r.reshape(N_CHANS, n_trials, n_t_r).transpose(1, 0, 2)  # 40 x 29 x n_t_r

    mi_start = int(MI_START_S * SFREQ_MODEL)
    mi_stop  = mi_start + WINDOW_SAMPLES
    assert mi_stop <= n_t_r, f"Window [{mi_start}:{mi_stop}] > resampled len {n_t_r}"

    X = data_trials[:, :, mi_start:mi_stop].astype(np.float32)
    assert X.shape == (40, N_CHANS, WINDOW_SAMPLES)
    assert np.isfinite(X).all()
    return X, y


def trial_zscore(X):
    """Per-trial, per-channel z-score. Safe to apply per-trial (no fitting)."""
    mu  = X.mean(axis=-1, keepdims=True)
    std = X.std(axis=-1, keepdims=True) + 1e-6
    return (X - mu) / std


def find_mat_files(root):
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"Data root not found: {root}")
    return sorted(root.rglob("*.mat"))


mat_files   = find_mat_files(DATA_ROOT)
all_sids    = sorted({subject_id_from_path(f) for f in mat_files})
SUBJECT_IDS = all_sids if CONFIG["subjects"] == "all" else sorted(int(s) for s in CONFIG["subjects"])
sid_to_path = {subject_id_from_path(f): f for f in mat_files if subject_id_from_path(f) in SUBJECT_IDS}

print(f"Found {len(mat_files)} .mat files, using {len(SUBJECT_IDS)} subjects")

## 5. EEGNet Model

In [ ]:
class EEGNet(nn.Module):
    """
    EEGNet (Lawhern et al., 2018) for binary EEG classification.
    Input: (batch, 1, n_chans, n_times)  — note the extra leading 1 (single EEG 'image').

    Architecture:
      Block 1: Temporal conv (F1 filters, kernel=sfreq//2) + Depthwise conv (D*F1 spatial filters)
      Block 2: Separable conv (F2=F1*D filters) + FC
    """
    def __init__(self, n_classes=2, n_chans=29, n_times=512,
                 F1=8, D=2, F2=None, dropout_rate=0.5, sfreq=128):
        super().__init__()
        F2 = F2 or F1 * D
        temporal_kernel = sfreq // 2  # 64 at 128 Hz

        # Block 1
        self.block1 = nn.Sequential(
            # Temporal conv: (B, 1, C, T) → (B, F1, C, T)
            nn.Conv2d(1, F1, kernel_size=(1, temporal_kernel), padding=(0, temporal_kernel // 2), bias=False),
            nn.BatchNorm2d(F1),
            # Depthwise conv over channels: (B, F1, C, T) → (B, F1*D, 1, T)
            nn.Conv2d(F1, F1 * D, kernel_size=(n_chans, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout_rate),
        )

        # Block 2
        self.block2 = nn.Sequential(
            # Separable conv: depthwise then pointwise
            nn.Conv2d(F1 * D, F1 * D, kernel_size=(1, 16), padding=(0, 8), groups=F1 * D, bias=False),
            nn.Conv2d(F1 * D, F2, kernel_size=(1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 8)),
            nn.Dropout(dropout_rate),
        )

        # Compute flatten size
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_chans, n_times)
            out   = self.block2(self.block1(dummy))
            self._flat_dim = int(np.prod(out.shape[1:]))

        self.classifier = nn.Linear(self._flat_dim, n_classes)

    def forward(self, x):
        # x: (B, C, T) → add channel dim → (B, 1, C, T)
        if x.ndim == 3:
            x = x.unsqueeze(1)
        x = self.block1(x)
        x = self.block2(x)
        x = x.flatten(1)
        return self.classifier(x)


# Smoke test
_model = EEGNet(n_chans=N_CHANS, n_times=WINDOW_SAMPLES,
                F1=CONFIG["F1"], D=CONFIG["D"],
                dropout_rate=CONFIG["dropout_rate"], sfreq=SFREQ_MODEL)
_x = torch.randn(4, N_CHANS, WINDOW_SAMPLES)
_out = _model(_x)
assert _out.shape == (4, 2), f"EEGNet output shape mismatch: {_out.shape}"
n_params = sum(p.numel() for p in _model.parameters())
print(f"EEGNet: {n_params:,} parameters, output shape: {_out.shape}")

## 6. Augmentation Helpers

In [ ]:
def augment_batch(X_batch, cfg_aug):
    """
    Apply light augmentation to a training batch (numpy, N x C x T).
    Only called during training. Never applied to test data.
    """
    if not cfg_aug.get("enabled", False):
        return X_batch

    X = X_batch.copy()

    # Gaussian noise
    noise_std_frac = cfg_aug.get("noise_std_fraction", 0.03)
    if noise_std_frac > 0:
        per_trial_std = X.std(axis=(-2, -1), keepdims=True) + 1e-6
        X = X + np.random.randn(*X.shape).astype(np.float32) * noise_std_frac * per_trial_std

    # Temporal jitter: roll each trial by a small random amount
    max_shift = cfg_aug.get("max_shift_samples", 6)
    if max_shift > 0:
        for i in range(len(X)):
            shift = np.random.randint(-max_shift, max_shift + 1)
            if shift != 0:
                X[i] = np.roll(X[i], shift, axis=-1)

    # Channel dropout: zero one channel with low probability
    ch_drop_p = cfg_aug.get("channel_dropout_prob", 0.05)
    if ch_drop_p > 0:
        for i in range(len(X)):
            if np.random.rand() < ch_drop_p:
                ch = np.random.randint(0, X.shape[1])
                X[i, ch, :] = 0.0

    return X


print("Augmentation helpers defined.")

## 7. Training Loop

In [ ]:
class TrialDataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(np.asarray(X, dtype=np.float32))
        self.y = torch.from_numpy(np.asarray(y, dtype=np.int64))
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]


def train_eegnet(X_train, y_train, cfg, device):
    """
    Train EEGNet on one training fold. Early stopping on internal 80/20 val split.
    Returns: trained model, training log.
    """
    # Internal 80/20 split for early stopping (never touches test fold)
    n = len(X_train)
    n_val = max(2, int(0.2 * n))
    idx = np.random.permutation(n)
    val_idx, tr_idx = idx[:n_val], idx[n_val:]

    X_tr, y_tr = X_train[tr_idx],  y_train[tr_idx]
    X_va, y_va = X_train[val_idx], y_train[val_idx]

    # Trial-wise z-score
    X_tr = trial_zscore(X_tr)
    X_va = trial_zscore(X_va)

    ds_tr = TrialDataset(X_tr, y_tr)
    ds_va = TrialDataset(X_va, y_va)
    dl_tr = torch.utils.data.DataLoader(ds_tr, batch_size=cfg["batch_size"], shuffle=True,
                                        drop_last=False)
    dl_va = torch.utils.data.DataLoader(ds_va, batch_size=cfg["batch_size"], shuffle=False)

    model = EEGNet(
        n_chans=N_CHANS, n_times=WINDOW_SAMPLES,
        F1=cfg["F1"], D=cfg["D"],
        dropout_rate=cfg["dropout_rate"], sfreq=SFREQ_MODEL,
    ).to(device)

    optimizer = optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    criterion = nn.CrossEntropyLoss()

    best_val_loss = float("inf")
    best_state    = copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})
    patience_ctr  = 0
    best_epoch    = 0
    aug_cfg       = cfg.get("augmentation", {})

    for epoch in range(cfg["max_epochs"]):
        model.train()
        for xb, yb in dl_tr:
            # Augmentation: apply to numpy before converting to tensor
            xb_np = augment_batch(xb.numpy(), aug_cfg)
            xb = torch.from_numpy(xb_np).to(device)
            yb = yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in dl_va:
                xb, yb = xb.to(device), yb.to(device)
                val_loss += criterion(model(xb), yb).item()
        val_loss /= max(len(dl_va), 1)

        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_state    = copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})
            patience_ctr  = 0
            best_epoch    = epoch
        else:
            patience_ctr += 1
            if patience_ctr >= cfg["patience"]:
                break

    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    model.eval()
    return model, {"best_epoch": best_epoch, "best_val_loss": best_val_loss, "stopped_epoch": epoch}


@torch.no_grad()
def predict(model, X_test, device, batch_size=16):
    """Run inference. X_test must be z-scored before calling."""
    ds  = TrialDataset(X_test, np.zeros(len(X_test), dtype=np.int64))
    dl  = torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=False)
    preds = []
    for xb, _ in dl:
        out = model(xb.to(device))
        preds.append(out.argmax(dim=-1).cpu().numpy())
    return np.concatenate(preds)


print("Training loop defined.")

## 8. Per-Subject Cross-Validation Runner

In [ ]:
def collapse_diagnostics(y_pred, n_classes=2):
    counts = np.bincount(y_pred, minlength=n_classes)
    dominant = counts.max() / counts.sum() if counts.sum() > 0 else 1.0
    return {
        "collapse_flag":  bool(dominant > 0.95),
        "collapse_ratio": float(dominant),
        "pred_counts":    counts.tolist(),
    }


def run_subject(sid, X, y, cfg, device):
    sss = StratifiedShuffleSplit(
        n_splits=cfg["n_repeats"],
        test_size=cfg["test_size"],
        random_state=cfg["random_state"],
    )
    fold_results = []
    train_logs   = []

    for fold_idx, (tr_idx, te_idx) in enumerate(sss.split(X, y)):
        X_train, X_test = X[tr_idx], X[te_idx]
        y_train, y_test = y[tr_idx], y[te_idx]

        assert len(np.unique(y_train)) == 2, f"Sub {sid} fold {fold_idx}: single class in train"

        try:
            model, log = train_eegnet(X_train, y_train, cfg, device)
            train_logs.append({"fold_id": fold_idx, **log})

            # Apply same z-score to test (per-trial, no fitting)
            X_test_z = trial_zscore(X_test)
            y_pred   = predict(model, X_test_z, device)

        except Exception as exc:
            print(f"  Sub {sid} fold {fold_idx} ERROR: {exc}")
            y_pred = np.zeros(len(y_test), dtype=int)
            log    = {"best_epoch": -1, "best_val_loss": float("nan"), "stopped_epoch": -1}
            train_logs.append({"fold_id": fold_idx, **log})

        acc  = float(accuracy_score(y_test, y_pred))
        bacc = float(balanced_accuracy_score(y_test, y_pred))
        cm   = confusion_matrix(y_test, y_pred, labels=[0, 1]).tolist()
        diag = collapse_diagnostics(y_pred)

        cm_arr = np.array(cm)
        left_recall  = cm_arr[0, 0] / cm_arr[0].sum() if cm_arr[0].sum() > 0 else float("nan")
        right_recall = cm_arr[1, 1] / cm_arr[1].sum() if cm_arr[1].sum() > 0 else float("nan")

        fold_results.append({
            "subject_id":        sid,
            "fold_id":           fold_idx,
            "accuracy":          acc,
            "balanced_accuracy": bacc,
            "left_recall":       float(left_recall),
            "right_recall":      float(right_recall),
            "confusion_matrix":  cm,
            "collapse_flag":     diag["collapse_flag"],
            "collapse_ratio":    diag["collapse_ratio"],
            "pred_counts":       diag["pred_counts"],
            "n_train":           int(len(y_train)),
            "n_test":            int(len(y_test)),
            "best_epoch":        log.get("best_epoch", -1),
            "stopped_epoch":     log.get("stopped_epoch", -1),
        })

    return fold_results, train_logs


print("Subject runner defined.")

## 9. Run All Subjects

In [ ]:
ALL_FOLD_RESULTS  = []
ALL_TRAIN_LOGS    = []
SUBJECT_SUMMARIES = []

print(f"Running {len(SUBJECT_IDS)} subjects | device={DEVICE}")
print("=" * 60)

for sid in SUBJECT_IDS:
    mat_path = sid_to_path.get(sid)
    if mat_path is None:
        print(f"  Sub {sid:02d}: no file, skipping")
        continue
    try:
        X, y = preprocess_subject(mat_path)
    except Exception as exc:
        print(f"  Sub {sid:02d}: preprocess error — {exc}")
        continue

    fold_res, train_logs = run_subject(sid, X, y, CONFIG, DEVICE)
    ALL_FOLD_RESULTS.extend(fold_res)
    ALL_TRAIN_LOGS.extend([{"subject_id": sid, **l} for l in train_logs])

    accs  = [r["accuracy"] for r in fold_res]
    baccs = [r["balanced_accuracy"] for r in fold_res]
    collapses = sum(1 for r in fold_res if r["collapse_flag"])
    mean_bacc = np.mean(baccs)

    SUBJECT_SUMMARIES.append({
        "subject_id":             sid,
        "mean_accuracy":          float(np.mean(accs)),
        "std_accuracy":           float(np.std(accs)),
        "mean_balanced_accuracy": float(mean_bacc),
        "std_balanced_accuracy":  float(np.std(baccs)),
        "n_folds":                len(fold_res),
        "n_collapsed_folds":      collapses,
        "mean_best_epoch":        float(np.mean([l["best_epoch"] for l in train_logs])),
    })

    print(f"  Sub {sid:02d}: bal_acc={mean_bacc*100:.1f}% ± {np.std(baccs)*100:.1f}%  "
          f"collapse={collapses}/{len(fold_res)}")

print("=" * 60)
print(f"Done. Total folds: {len(ALL_FOLD_RESULTS)}")

## 10. Aggregate Results

In [ ]:
fold_df    = pd.DataFrame(ALL_FOLD_RESULTS)
subject_df = pd.DataFrame(SUBJECT_SUMMARIES)

all_baccs = fold_df["balanced_accuracy"].values
all_accs  = fold_df["accuracy"].values
n_collapsed = int(fold_df["collapse_flag"].sum())

cm_total = np.zeros((2, 2), dtype=int)
for row in ALL_FOLD_RESULTS:
    cm_total += np.array(row["confusion_matrix"])

left_recall_agg  = cm_total[0, 0] / cm_total[0].sum() if cm_total[0].sum() > 0 else float("nan")
right_recall_agg = cm_total[1, 1] / cm_total[1].sum() if cm_total[1].sum() > 0 else float("nan")

global_summary = {
    "method":                 "eegnet",
    "n_subjects":             len(SUBJECT_SUMMARIES),
    "n_folds_total":          len(ALL_FOLD_RESULTS),
    "mean_accuracy":          float(np.mean(all_accs)),
    "std_accuracy":           float(np.std(all_accs)),
    "mean_balanced_accuracy": float(np.mean(all_baccs)),
    "std_balanced_accuracy":  float(np.std(all_baccs)),
    "n_collapsed_folds":      n_collapsed,
    "collapse_rate":          float(n_collapsed / max(len(ALL_FOLD_RESULTS), 1)),
    "left_recall_agg":        float(left_recall_agg),
    "right_recall_agg":       float(right_recall_agg),
    "confusion_matrix":       cm_total.tolist(),
}

print("=" * 60)
print("GLOBAL RESULTS — EEGNet")
print(f"  Mean Balanced Accuracy: {global_summary['mean_balanced_accuracy']*100:.2f}% ± {global_summary['std_balanced_accuracy']*100:.2f}%")
print(f"  Mean Accuracy:          {global_summary['mean_accuracy']*100:.2f}% ± {global_summary['std_accuracy']*100:.2f}%")
print(f"  Left recall:  {left_recall_agg*100:.1f}%  |  Right recall: {right_recall_agg*100:.1f}%")
print(f"  Collapsed folds: {n_collapsed}/{len(ALL_FOLD_RESULTS)} ({global_summary['collapse_rate']*100:.1f}%)")
print(f"  Aggregated CM: {cm_total.tolist()}")
print("=" * 60)

## 11. Save Artifacts

In [ ]:
fold_df.to_csv(ARTIFACT_ROOT / "fold_results.csv", index=False)
subject_df.to_csv(ARTIFACT_ROOT / "subject_summary.csv", index=False)
pd.DataFrame(ALL_TRAIN_LOGS).to_csv(ARTIFACT_ROOT / "training_logs.csv", index=False)

with open(ARTIFACT_ROOT / "global_summary.json", "w") as f:
    json.dump(global_summary, f, indent=2)
with open(ARTIFACT_ROOT / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2, default=str)

print("Saved: fold_results.csv, subject_summary.csv, training_logs.csv, global_summary.json")

## 12. Plots

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(4, 3))
im = ax.imshow(cm_total, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Pred Left", "Pred Right"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["True Left", "True Right"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm_total[i, j]), ha="center", va="center", fontsize=12)
ax.set_title("Aggregated CM — EEGNet")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(ARTIFACT_ROOT / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

# Per-subject accuracy
fig, ax = plt.subplots(figsize=(max(8, len(subject_df) * 0.4), 4))
sids  = subject_df["subject_id"].values
baccs = subject_df["mean_balanced_accuracy"].values * 100
stds  = subject_df["std_balanced_accuracy"].values * 100
ax.bar(sids, baccs, yerr=stds, capsize=3, color="seagreen", alpha=0.8)
ax.axhline(50, color="red", linestyle="--", label="Chance")
ax.axhline(float(np.mean(baccs)), color="orange", linestyle="-", label=f"Mean={np.mean(baccs):.1f}%")
ax.set_xlabel("Subject ID")
ax.set_ylabel("Balanced Accuracy (%)")
ax.set_title("Per-subject balanced accuracy — EEGNet")
ax.legend()
ax.set_xticks(sids)
ax.set_xticklabels(sids, rotation=90, fontsize=7)
plt.tight_layout()
plt.savefig(ARTIFACT_ROOT / "subject_accuracy_plot.png", dpi=150, bbox_inches="tight")
plt.show()

# Early stopping epochs
log_df = pd.DataFrame(ALL_TRAIN_LOGS)
if "best_epoch" in log_df.columns:
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.hist(log_df["best_epoch"].dropna(), bins=20, color="steelblue", alpha=0.8)
    ax.set_xlabel("Best epoch")
    ax.set_ylabel("Count")
    ax.set_title("EEGNet best epoch distribution across folds")
    plt.tight_layout()
    plt.savefig(ARTIFACT_ROOT / "best_epoch_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()

print("Plots saved.")